# User-facing control that affects LLM Experimentation
(e.g., a slider for response verbosity, a dropdown to switch response style, or a toggle to restrict scope).

Your repository should contain a notebook with experiments and narrative, and the resulting option and summary of motivation should be reflected in the specifications document.

## Imports

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
from chatlas import ChatAnthropic

## Load, merge, and view data

In [4]:
FEATURES_PATH = Path("../data/raw/ai_productivity_features.csv")
TARGETS_PATH = Path("../data/raw/ai_productivity_targets.csv")

features = pd.read_csv(FEATURES_PATH)
targets = pd.read_csv(TARGETS_PATH)

df = features.merge(targets, on="Employee_ID", how="left")
df.head()

,Employee_ID,job_role,experience_years,ai_tool_usage_hours_per_week,tasks_automated_percent,manual_work_hours_per_week,learning_time_hours_per_week,deadline_pressure_level,meeting_hours_per_week,collaboration_hours_per_week,error_rate_percent,task_complexity_score,focus_hours_per_day,work_life_balance_score,burnout_risk_score,productivity_score,burnout_risk_level
0,3c6ca882-3fa3-446b-8208-c92f3f306f06,Writer,19,11.8,28.5,19.2,1.4,High,1.9,2.3,0.20,2,7.1,4.8,10.00,81.0,High
1,02f168cc-7747-4dbd-a868-ea2cfb41e22a,Designer,4,10.8,24.1,23.3,2.6,Low,8.0,9.8,1.82,3,3.4,5.5,6.78,59.2,Medium
2,d39ce8c9-6e2a-4f86-b888-e2b5f4a18cf7,Developer,6,25.9,69.4,10.0,1.4,Medium,6.8,8.9,5.52,5,4.6,3.8,9.66,62.4,High
3,14511660-d78a-453f-9449-f17cd239ec27,Manager,20,7.9,17.2,25.1,0.2,High,3.5,8.6,1.14,5,5.6,3.9,10.00,76.8,High
4,0597f0bb-ed5a-4e35-94ac-3f0f6a5c2bc2,Developer,15,8.6,20.6,20.1,1.4,Low,5.9,5.3,2.75,10,1.0,7.4,5.38,53.7,Medium


## Define the dataset description

In [5]:
data_description = """
    Employee-level workplace wellbeing and productivity dataset.
    
    Each row represents one employee.
    
    Columns:
    - Employee_ID: unique identifier for each employee.
    - job_role: employee job role/category.
    - experience_years: years of experience.
    - ai_tool_usage_hours_per_week: hours per week spent using AI tools.
    - tasks_automated_percent: percent of tasks automated with AI/tools.
    - manual_work_hours_per_week: hours per week spent on manual work.
    - learning_time_hours_per_week: hours per week spent learning new tools/workflows.
    - deadline_pressure_level: categorical deadline pressure level (Low, Medium, High).
    - meeting_hours_per_week: hours per week spent in meetings.
    - collaboration_hours_per_week: hours per week spent collaborating.
    - error_rate_percent: percentage error rate.
    - task_complexity_score: task complexity score.
    - focus_hours_per_day: average focus/deep work hours per day.
    - work_life_balance_score: numeric work-life balance score.
    - burnout_risk_score: numeric burnout risk score.
    - productivity_score: numeric productivity score.
    - burnout_risk_level: burnout category label.
    
    Use this dataset to analyze:
    - burnout risk by role
    - relationships between AI usage and burnout
    - productivity vs burnout
    - work-life balance patterns
    - workload-related drivers of burnout
    
    Important:
    - Stay grounded in the available dataset columns.
    - Do not make causal claims.
    - Frame findings as associations, patterns, or comparisons.
"""

## Create a compact dataset summary for the prompt

This helps the model stay grounded.

In [10]:
def build_dataset_snapshot(df: pd.DataFrame) -> str:
    role_counts = df["job_role"].value_counts().to_dict()
    burnout_counts = df["burnout_risk_level"].value_counts().to_dict()

    summary = {
        "n_rows": int(len(df)),
        "job_roles": role_counts,
        "burnout_levels": burnout_counts,
        "avg_ai_usage_hours": round(
            df["ai_tool_usage_hours_per_week"].mean(),
            2
        ),
        "avg_manual_hours": round(
            df["manual_work_hours_per_week"].mean(), 2),
        "avg_burnout_score": round(
            df["burnout_risk_score"].mean(), 
            2
        ),
        "avg_productivity_score": round(df["productivity_score"].mean(), 2),
        "avg_work_life_balance": round(
            df["work_life_balance_score"].mean(),
            2
        ),
    }

    lines = ["Dataset snapshot:"]
    for k, v in summary.items():
        lines.append(f"- {k}: {v}")
    return "\n".join(lines)

dataset_snapshot = build_dataset_snapshot(df)
print(dataset_snapshot)

Dataset snapshot:
- n_rows: 4500
- job_roles: {'Developer': 1115, 'Analyst': 892, 'Designer': 722, 'Marketer': 658, 'Manager': 652, 'Writer': 461}
- burnout_levels: {'High': 3303, 'Medium': 1087, 'Low': 110}
- avg_ai_usage_hours: 10.35
- avg_manual_hours: 22.37
- avg_burnout_score: 8.35
- avg_productivity_score: 64.95
- avg_work_life_balance: 4.72


## Define the style instructions

In [11]:
STYLE_INSTRUCTIONS = {
    "executive": 
    """
    Respond in an Executive Summary style.
    - Use plain, concise language.
    - Emphasize the main takeaway first.
    - Mention practical workplace implications.
    - Avoid technical jargon unless necessary.
    - Keep the answer brief and decision-oriented.
    """,
    "analytical": 
    """
    Respond in an Analytical Explanation style.
    - Explain the main patterns clearly.
    - Compare relevant groups or variables where useful.
    - Balance clarity and detail.
    - Keep the answer grounded in the dataset.
    - Do not overstate conclusions.
    """,
    "technical": """
    Respond in a Technical Interpretation style.
    - Refer explicitly to dataset variables where relevant.
    - Emphasize associations, not causality.
    - Mention limitations or ambiguity when appropriate.
    - Use more precise analytical language.
    - Keep the answer dataset-grounded and method-aware.
    """
}

## Define the evaluation questions

In [6]:
questions = [
    "Which job roles appear to have the highest burnout risk?",
    "How does AI tool usage relate to burnout risk in this dataset?",
    "Show employees with high burnout risk and low productivity, and summarize what they have in common.",
    "What workplace factors seem most associated with burnout in this dataset?"
]

## Initialize the model

In [7]:
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

if not anthropic_key:
    raise ValueError("ANTHROPIC_API_KEY is not set in your environment.")

chat_model = ChatAnthropic(model="claude-sonnet-4-0")

## Build a reusable prompt function

In [8]:
def build_prompt(question: str, style_key: str, data_description: str, dataset_snapshot: str) -> str:
    return f"""
    You are an AI assistant for a workplace burnout analytics dashboard.
    
    {STYLE_INSTRUCTIONS[style_key]}
    
    Context about the dataset:
    {data_description}
    
    {dataset_snapshot}
    
    User question:
    {question}
    
    Instructions:
    - Answer using only the dataset context provided.
    - If the question suggests causality, reframe it as association.
    - If a precise subgroup is requested, describe how you would identify it from the dataset fields.
    - Be truthful about uncertainty.
    """

## Run the experiment

This generates one response per question per style.

In [12]:
def ask_model(question: str, style_key: str) -> str:
    prompt = build_prompt(
        question=question,
        style_key=style_key,
        data_description=data_description,
        dataset_snapshot=dataset_snapshot,
    )
    response = chat_model.chat(prompt)
    return str(response)

records = []

for question in questions:
    for style_key in ["executive", "analytical", "technical"]:
        print(f"Running: {style_key} | {question}")
        answer = ask_model(question, style_key)
        records.append(
            {
                "question": question,
                "style": style_key,
                "response": answer,
            }
        )

experiment_df = pd.DataFrame(records)
experiment_df

Running: executive | Which job roles appear to have the highest burnout risk?


<br>

## Executive Summary: Burnout Risk by Job Role

**Key Finding:** Based on the dataset of 4,500 employees, we need to analyze burnout patterns across the six job roles (Developer, Analyst, Designer, Marketer, Manager, Writer) to identify which roles show the highest risk levels.

**Analysis Approach:** To determine role-specific burnout risk, we should examine:
- Average burnout risk scores by job role
- Distribution of burnout risk levels (High/Medium/Low) within each role
- Comparison of work-life balance scores across roles

**Current Dataset Context:** 
- Overall burnout levels are concerning: 73% of employees show High burnout risk, 24% Medium, and only 2% Low
- Average burnout score is 8.35 (on whatever scale is used)
- Work-life balance averages 4.72 across all roles

**Workplace Implications:** Once we identify the highest-risk roles, leadership should:
- Prioritize intervention resources for those specific job functions
- Investigate role-specific workload patterns (manual hours, meeting time, deadline pressure)
- Consider whether certain roles need structural changes to workload or expectations

**Next Step Required:** The specific breakdown of burnout metrics by individual job role is needed to definitively answer which roles show the highest risk patterns. This analysis would reveal whether technical roles (Developers, Analysts) or leadership roles (Managers) face greater burnout challenges.

Running: analytical | Which job roles appear to have the highest burnout risk?


<br>

## Burnout Risk Patterns by Job Role

Based on the dataset structure, I can outline how to identify roles with the highest burnout risk, though the specific role-level breakdowns aren't provided in the current snapshot.

### Analysis Framework

To determine which job roles show the highest burnout risk, we would need to examine the distribution of burnout metrics across the six roles in the dataset:

**Primary Indicators:**
- Average burnout risk scores by role (comparing against the overall average of 8.35)
- Proportion of employees in each "High" burnout risk category within each role
- Work-life balance score variations by role (compared to the 4.72 overall average)

**Supporting Workload Patterns:**
The dataset allows us to explore role-specific patterns that associate with burnout risk:
- Manual work hours per week (overall average: 22.37 hours)
- Meeting and collaboration time commitments
- Deadline pressure levels by role
- Task complexity scores
- Focus hours available per day

### What We Know About the Overall Population

The dataset reveals concerning burnout levels across all roles:
- 73% of employees fall into the "High" burnout risk category
- Only 2.4% show "Low" burnout risk
- This suggests burnout is a widespread organizational issue rather than isolated to specific roles

### Role Comparison Approach

To identify the highest-risk roles, we would compare:

1. **Developers (1,115 employees)** - The largest group, likely with high task complexity and focus requirements
2. **Analysts (892 employees)** - Second largest group, potentially dealing with data-intensive work
3. **Managers (652 employees)** - May show different patterns due to meeting loads and collaboration demands
4. **Other roles** (Designer, Marketer, Writer) - Smaller groups that might reveal distinct burnout patterns

### Limitations

Without the role-specific breakdown of burnout scores and risk levels, I cannot definitively state which roles show the highest risk. The analysis would require aggregating the burnout metrics by job_role to reveal these patterns clearly.

The high overall burnout prevalence (73% High risk) suggests that multiple roles likely face significant burnout challenges, making the specific role rankings particularly important for targeted interventions.

Running: technical | Which job roles appear to have the highest burnout risk?


...

Running: analytical | What workplace factors seem most associated with burnout in this dataset?


<br>

## Workplace Factors Associated with Burnout Risk

### Overview of Burnout Patterns

The dataset reveals an organization in crisis, with 73% of employees classified as high burnout risk and only 2.4% showing low risk. This extreme distribution suggests that burnout associations may be widespread across multiple workplace factors rather than isolated to specific conditions.

### Key Factor Categories for Analysis

**Workload and Time Management Variables**
Several variables in the dataset can reveal workload-related burnout patterns:
- `manual_work_hours_per_week` (currently averaging 22.37 hours) may show higher values among high-burnout employees
- `focus_hours_per_day` could be reduced in burnout cases due to fragmented attention
- The balance between `ai_tool_usage_hours_per_week` (average 10.35) and manual work may indicate efficiency issues
- `tasks_automated_percent` might be lower among employees struggling with burnout

**Pressure and Complexity Factors**
- `deadline_pressure_level` likely shows strong associations, with high-burnout employees experiencing more "High" pressure classifications
- `task_complexity_score` may correlate with burnout, though the relationship could vary by role
- `error_rate_percent` could both contribute to and result from burnout stress

**Social and Collaboration Patterns**
- `meeting_hours_per_week` might show problematic patterns if excessive meeting loads associate with burnout
- `collaboration_hours_per_week` could reveal whether isolation or over-collaboration correlates with stress levels

### Work-Life Balance as a Central Factor

The `work_life_balance_score` (averaging 4.72) likely represents one of the strongest burnout predictors. Employees with scores significantly below this average may show disproportionately higher burnout risk scores. This variable could serve as both an outcome measure and a contributing factor to the burnout patterns observed.

### Role-Specific Associations

Given the varied job roles and sample sizes, burnout factors likely differ across functions:

**Technical Roles (Developer: 1,115; Analyst: 892)**
These roles may show associations between burnout and technical complexity, learning overhead (`learning_time_hours_per_week`), or difficulty automating complex tasks.

**Management Roles (652 employees)**
Managers might exhibit different patterns, potentially showing associations with high meeting loads, collaboration demands, or the stress of managing others during organizational burnout.

**Creative and Communication Roles (Designer, Marketer, Writer)**
These smaller groups may reveal distinct patterns related to deadline pressure, creative complexity, or different automation challenges.

### Comparative Analysis Framework

To identify the strongest associations, we would need to compare burnout scores across different levels of each factor:

**High vs. Low Comparisons:**
- Employees with high manual work hours vs. those with more automated workflows
- High deadline pressure vs. low pressure employees
- Above-average vs. below-average work-life balance scores
- Different ranges of meeting hours and collaboration time

**Interaction Effects:**
Some factors may compound each other. For example, high manual work combined with high deadline pressure might show stronger burnout associations than either factor alone.

### Limitations in Current Analysis

The provided snapshot shows overall averages but doesn't reveal the correlations or group comparisons needed to definitively rank factor importance. The extreme skew toward high burnout (73%) also means that most workplace conditions in this organization may associate with elevated burnout risk.

Additionally, the cross-sectional nature means we cannot determine whether certain factors precede burnout development or result from it. For instance, reduced focus hours might cause increased stress, or existing burnout might impair concentration abilities.

### Expected Patterns

Based on the dataset structure and organizational research, the strongest burnout associations likely involve work-life balance scores, deadline pressure levels, and the ratio of manual to automated work. However, given the severity of the organization-wide burnout crisis, multiple factors probably contribute simultaneously to the elevated risk levels observed across the workforce.

Running: technical | What workplace factors seem most associated with burnout in this dataset?


<br>

## Technical Analysis: Workplace Factor Associations with Burnout Risk

### Statistical Framework for Association Analysis

**Primary Dependent Variables:**
- `burnout_risk_score` (continuous, μ = 8.35)
- `burnout_risk_level` (categorical: High n=3303, Medium n=1087, Low n=110)

**Independent Variable Categories:**
The dataset structure permits examination of associations across four domains:

### Workload Configuration Variables

**Continuous Predictors:**
- `manual_work_hours_per_week` (μ = 22.37): Expected negative association with wellbeing
- `ai_tool_usage_hours_per_week` (μ = 10.35): Association direction uncertain without analysis
- `tasks_automated_percent`: Hypothesized inverse relationship with burnout
- `focus_hours_per_day`: Likely negative correlation with `burnout_risk_score`

**Expected Analysis Method:** Pearson correlations with `burnout_risk_score`, followed by ANOVA comparing means across `burnout_risk_level` strata.

### Environmental Stressor Variables

**Categorical Predictor:**
- `deadline_pressure_level`: Ordinal variable requiring chi-square analysis against `burnout_risk_level` and one-way ANOVA for `burnout_risk_score` differences across pressure categories

**Continuous Stress Indicators:**
- `meeting_hours_per_week` and `collaboration_hours_per_week`: Potential curvilinear relationships requiring quadratic modeling
- `error_rate_percent`: Bidirectional association possible (stress → errors, errors → stress)
- `task_complexity_score`: May interact with `experience_years`

### Work-Life Integration Metrics

**Primary Wellbeing Indicator:**
- `work_life_balance_score` (μ = 4.72): Likely strongest negative correlation with burnout variables
- Expected effect size: r > -0.40 based on organizational psychology literature

### Methodological Considerations

**Distribution Characteristics:**
The extreme skew in `burnout_risk_level` (73.4% High, 24.2% Medium, 2.4% Low) creates analytical constraints:
- Low statistical power for detecting differences in the "Low" burnout stratum
- Potential ceiling effects in association strength estimation
- May require logistic regression comparing High vs. Medium+Low categories

**Multicollinearity Assessment Required:**
Several predictor variables likely exhibit intercorrelation:
- `manual_work_hours_per_week` and `tasks_automated_percent` (expected r < -0.30)
- `ai_tool_usage_hours_per_week` and `learning_time_hours_per_week` (adaptation overhead)
- Meeting and collaboration variables with `job_role` stratification effects

### Role-Stratified Analysis Requirements

**Interaction Effects:**
Association patterns may vary significantly across `job_role` categories given unequal sample sizes and functional differences:
- Manager role (n=652): Different stress profile from individual contributors
- Technical roles (Developer n=1115, Analyst n=892): Automation variables may show stronger associations
- Creative roles (Designer, Writer): Task complexity relationships may differ

### Expected Association Ranking

**Hypothesized Strongest Predictors (based on dataset structure):**
1. `work_life_balance_score`: Direct wellbeing measure
2. `deadline_pressure_level`: Categorical stressor with clear ordinality
3. `manual_work_hours_per_week`: Workload inefficiency indicator
4. `focus_hours_per_day`: Cognitive resource availability

**Interaction Terms to Test:**
- `deadline_pressure_level × task_complexity_score`
- `manual_work_hours_per_week × tasks_automated_percent`
- `job_role × meeting_hours_per_week`

### Analytical Limitations

**Cross-Sectional Constraints:**
Temporal precedence cannot be established. Variables showing strong associations with `burnout_risk_score` may represent:
- Antecedent conditions contributing to burnout development
- Consequences of existing burnout states
- Bidirectional relationships with feedback loops

**Missing Variance Parameters:**
Without standard deviations for continuous variables, effect size estimation and statistical significance testing cannot be performed on the current dataset snapshot.

**Potential Confounding:**
Organizational-level factors (team assignment, management quality, workload distribution policies) not captured in individual-level variables may inflate or attenuate observed associations.

### Model Specification Recommendations

**Primary Analysis:** Multiple regression with `burnout_risk_score` as dependent variable, including main effects and theoretically justified interaction terms.

**Secondary Analysis:** Multinomial logistic regression predicting `burnout_risk_level` categories, though the extreme class imbalance may require collapsed categories or resampling techniques.

The identification of strongest workplace factor associations requires correlation matrix examination and regression modeling not available in the current dataset summary.

,question,style,response
0,Which job roles appear to have the highest bur...,executive,## Executive Summary: Burnout Risk by Job Role...
1,Which job roles appear to have the highest bur...,analytical,## Burnout Risk Patterns by Job Role\n\nBased ...
2,Which job roles appear to have the highest bur...,technical,## Technical Analysis: Job Role and Burnout Ri...
3,How does AI tool usage relate to burnout risk ...,executive,## Executive Summary: AI Tool Usage and Burnou...
4,How does AI tool usage relate to burnout risk ...,analytical,## AI Tool Usage and Burnout Risk Associations...
5,How does AI tool usage relate to burnout risk ...,technical,## Technical Analysis: AI Tool Usage and Burno...
6,Show employees with high burnout risk and low ...,executive,"## Executive Summary: High Burnout, Low Produc..."
7,Show employees with high burnout risk and low ...,analytical,"## Analysis: High Burnout Risk, Low Productivi..."
8,Show employees with high burnout risk and low ...,technical,"## Technical Analysis: High Burnout Risk, Low ..."
9,What workplace factors seem most associated wi...,executive,## Executive Summary: Key Workplace Factors As...


## Save the experiment results

In [13]:
output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

experiment_path = output_dir / "querychat_response_style_experiments.csv"
experiment_df.to_csv(experiment_path, index=False)

experiment_path

WindowsPath('notebooks/outputs/querychat_response_style_experiments.csv')

## Add a simple manual evaluation table

In [14]:
evaluation_template = experiment_df.copy()
evaluation_template["relevance_score"] = None
evaluation_template["clarity_score"] = None
evaluation_template["actionability_score"] = None
evaluation_template["audience_fit_score"] = None
evaluation_template["notes"] = ""

evaluation_path = output_dir / "querychat_response_style_evaluation_template.csv"
evaluation_template.to_csv(evaluation_path, index=False)

evaluation_template.head()

,question,style,response,relevance_score,clarity_score,actionability_score,audience_fit_score,notes
0,Which job roles appear to have the highest bur...,executive,## Executive Summary: Burnout Risk by Job Role...,None,None,None,None,
1,Which job roles appear to have the highest bur...,analytical,## Burnout Risk Patterns by Job Role\n\nBased ...,None,None,None,None,
2,Which job roles appear to have the highest bur...,technical,## Technical Analysis: Job Role and Burnout Ri...,None,None,None,None,
3,How does AI tool usage relate to burnout risk ...,executive,## Executive Summary: AI Tool Usage and Burnou...,None,None,None,None,
4,How does AI tool usage relate to burnout risk ...,analytical,## AI Tool Usage and Burnout Risk Associations...,None,None,None,None,
